# Source-bound ViT-equivalent geometry runner

Purpose: prepare, checkpoint and exactly resume paired full-JEPA experiments. Inputs: registered plan, train-only data bundle and independent development membership. Outputs: immutable initial models, full-state optimizer checkpoints and paired milestones. Definitions import in seconds and never train. Main front door: `notebooks/15_ViT_Equivalent_Geometry.ipynb`; resume cell `launch-vit-geometry`. Checkpoints: `checkpoints/<run_id>/stages/<stage_id>/arms/<arm_id>/`. Resume in a fresh kernel with identical source/plan/data/device environment, Run All; deadlines never reset. Long training requires detached tmux and passed actual-config4versus2+2 full-state smoke. The inherited checkpoint machinery retains latest two and best plus immutable evaluation milestones, with one writer and interruption-safe writes. Later backend hooks must bind their notebook export hashes before registration.

In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
from pathlib import Path
import json
ROOT=Path(os.environ.get('FMM_JEPA_ROOT','/home/benjamin/workspace/ahr_go/fmm_jeppa')).resolve()
_boot={'__file__':str(ROOT/'fmm_jepa/bootstrap.ipynb')}
for _cell in json.loads((ROOT/'fmm_jepa/bootstrap.ipynb').read_text())['cells']:
    if _cell['cell_type']=='code' and 'ahr-export' in _cell.get('metadata',{}).get('tags',[]):
        exec(compile(''.join(_cell['source']),'bootstrap#ahr-export','exec'),_boot)
_boot['install_notebook_package'](ROOT)


In [ ]:
"""Source-bound paired ViT/optional geometry runner. Definitions never train."""
from pathlib import Path
import copy,gc,hashlib,json,os,sys,time,types
import torch
from fmm_jepa.training import atomic_json,digest_file,definition_hashes
from fmm_jepa.food101_training import isolated_rng,_hash
import fmm_jepa.semantic_pilot as _pilot
ROOT=Path(globals().get('__file__',ROOT/'experiments/15_vit_geometry/runner.ipynb')).resolve().parents[2]
RESUME_NOTEBOOK='notebooks/15_ViT_Equivalent_Geometry.ipynb'
RESUME_CELL='launch-vit-geometry'
VARIANTS=('vit','optional_geometry')
_MODEL_BUILDER_OVERRIDE=None
_BOUND_SOURCE_IDENTITY={}

def load_tagged(root,relative,name):
    path=(Path(root)/relative).resolve()
    if name in sys.modules:
        module=sys.modules[name]
        if Path(module.__file__).resolve()!=path:raise ValueError('Module source path changed')
        return module
    module=types.ModuleType(name);module.__file__=str(path);sys.modules[name]=module
    cells=json.loads(path.read_text())['cells'];selected=[c for c in cells if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[])]
    if not selected:raise ValueError('Notebook has no explicit reusable cells')
    try:
        for index,cell in enumerate(selected):exec(compile(''.join(cell['source']),str(path)+'#ahr-export-'+str(index),'exec'),module.__dict__)
    except BaseException:sys.modules.pop(name,None);raise
    return module

_BASE=load_tagged(ROOT,'experiments/05_scaling/runner.ipynb','fmm15_isolated_inherited_runner05')
_BASE_ORIGINAL_HASHES=_BASE.stage_definition_hashes
normalize_stage_plan=_BASE.normalize_stage_plan

def export_hash(path):
    cells=json.loads(Path(path).read_text())['cells'];sources=[''.join(c['source']) for c in cells if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[])]
    if not sources:raise ValueError('Explicit ahr-export definitions required:'+str(path))
    return hashlib.sha256('\n'.join(sources).encode()).hexdigest()

def bind_model_builder(builder,source_identity):
    """Bind a later notebook backend with checked path->ahr-export hash identity.
    This does not modify15source. Rebind before loading any checkpoint/protocol.
    Existing exact/fresh scientific plans remain incompatible with a different hook.
    """
    global _MODEL_BUILDER_OVERRIDE,_BOUND_SOURCE_IDENTITY
    if not callable(builder) or not isinstance(source_identity,dict) or not source_identity:raise ValueError('Callable builder and nonempty notebook source hashes required')
    for relative,expected in source_identity.items():
        path=(ROOT/relative).resolve()
        if not path.is_relative_to(ROOT) or path.suffix!='.ipynb' or export_hash(path)!=expected:raise ValueError('Bound backend source hash differs:'+relative)
    _MODEL_BUILDER_OVERRIDE=builder;_BOUND_SOURCE_IDENTITY=dict(source_identity)
    return dict(_BOUND_SOURCE_IDENTITY)

def stage_definition_hashes(root):
    root=Path(root);result=_BASE_ORIGINAL_HASHES(root)
    for folder in ('experiments/11_manifold_long','experiments/15_vit_geometry'):
        for path in sorted((root/folder).glob('*.ipynb')):result[str(path.relative_to(root))]=export_hash(path)
    for relative,expected in _BOUND_SOURCE_IDENTITY.items():
        if export_hash(root/relative)!=expected:raise ValueError('Bound backend changed:'+relative)
        if relative in result and result[relative]!=expected:raise ValueError('Conflicting source binding')
        result[relative]=expected
    return result

def configure_environment(config):
    if os.environ.get('CUBLAS_WORKSPACE_CONFIG')!=':4096:8':raise RuntimeError('Restart fresh kernel with CUBLAS_WORKSPACE_CONFIG=:4096:8')
    if config.get('dtype','float32')!='float32':raise ValueError('FP32 only')
    torch.set_num_threads(int(config.get('cpu_threads',2)));torch.use_deterministic_algorithms(True)
    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
    if config.get('evaluation',{}).get('heldout_enabled',False):raise ValueError('Official test remains closed')
    device=torch.device(config.get('device','cpu'))
    if device.type=='cuda':torch.cuda.set_device(device)
    return {'device':str(device),'cpu_threads':torch.get_num_threads(),'deterministic_algorithms':torch.are_deterministic_algorithms_enabled(),
            'tf32_matmul':torch.backends.cuda.matmul.allow_tf32,'tf32_cudnn':torch.backends.cudnn.allow_tf32}

def configure_input_pipeline(config):
    workers=int(config.get('input_pipeline',{}).get('workers',0))
    if workers<0:raise ValueError('Nonnegative ordered input workers required')
    if workers:return load_tagged(ROOT,'experiments/11_manifold_long/input_pipeline.ipynb','fmm15_input_pipeline11').install_input_pipeline(workers=workers)
    return {'status':'SERIAL','workers':0}

def _default_build_model(config,kind):
    options=copy.deepcopy(config.get('kernel15',{}));variant=options.get('variant')
    if variant not in VARIANTS or kind!=('vit' if variant=='vit' else 'semantic'):raise ValueError('Explicit kind/variant mapping required')
    backend=load_tagged(ROOT,'experiments/15_vit_geometry/encoder.ipynb','fmm15_encoder')
    model_config=copy.deepcopy(config['model']);model_config['kernel15']=options
    return backend.build_model(model_config,kind,variant)

def build_model(config,kind):
    configure_input_pipeline(config)
    if config.get('execution_optimization',False):load_tagged(ROOT,'experiments/11_manifold_long/engineering.ipynb','fmm15_engineering11').install_execution_optimizations(ROOT)
    return (_MODEL_BUILDER_OVERRIDE or _default_build_model)(config,kind)

def initialize_models(config,artifact):
    from fmm_jepa.food101_evaluation import encoder_state_hash
    backend=load_tagged(ROOT,'experiments/15_vit_geometry/encoder.ipynb','fmm15_encoder')
    variant=config.get('kernel15',{}).get('variant')
    if variant not in VARIANTS:raise ValueError('Explicit kernel15 variant required')
    folder=Path(artifact)/'initialization';folder.mkdir(parents=True,exist_ok=True);path=folder/'manifest.json'
    identity=_hash({'model':config['model'],'kernel15':config['kernel15'],'seed':config['seeds']['model_init'],
        'bound_builder_sources':_BOUND_SOURCE_IDENTITY,'initialization_policy':'all-actual-vit-scaffold-qkv-projections-predictor-exact/v1'})
    baseline=copy.deepcopy(config);baseline['kernel15']['variant']='vit'
    with isolated_rng(config['seeds']['model_init']):
        models={'vit':_default_build_model(baseline,'vit').cpu()}
        if variant!='vit':models['semantic']=build_model(config,'semantic').cpu()
    if path.exists():
        info=json.loads(path.read_text())
        if info['identity']!=identity or set(info['states'])!=set(models):raise ValueError('Initialization binding differs; refusing restart')
        for kind,model in models.items():
            for record in info['states'][kind].values():
                if digest_file(record['path'])!=record['sha256']:raise ValueError('Initial-state checksum mismatch')
            model.load_state_dict(torch.load(info['states'][kind]['model']['path'],map_location='cpu',weights_only=True))
        return models,info
    if list(folder.glob('*.pt')):raise ValueError('Orphan initial states without manifest; refusing overwrite')
    source=models['vit'];copied=['context.'+name for name in source.context.state_dict()]+['predictor.'+name for name in source.predictor.state_dict()]
    if variant!='vit':backend.initialize_from_vit(models['semantic'],source)
    records={};counts={};shared={}
    for kind,model in models.items():
        model.target.load_state_dict(model.context.state_dict());model.target.requires_grad_(False).eval()
        counts[kind]=backend.parameter_counts(model);records[kind]={}
        shared[kind]=encoder_state_hash({name:model.state_dict()[name].detach().cpu() for name in copied})
        for label,state in [('model',model.state_dict()),('teacher',model.target.state_dict())]:
            file=folder/f'{kind}_initial_{label}.pt';_atomic_torch(file,state);records[kind][label]=_record(file)
    if len(set(shared.values()))!=1:raise AssertionError('Every actual ViT shared parameter must match')
    info={'identity':identity,'variant':variant,'states':records,'parameter_counts':counts,'copied_components':copied,
        'shared_parameter_hashes':shared,'kernel_mixer_shared_sha256':None,'inactive_context_scaffold_parameters':{k:0 for k in models},
        'independent_storage_verified':True,'bound_builder_sources':dict(_BOUND_SOURCE_IDENTITY),
        'comparison':'Every actual ViT stem/block norm/MLP/QKV/output projection/final norm/predictor tensor exactly copied. Raw Q/K remain independent. Extra signed spatial bias starts0; independent temperature multiplier starts1.',
        'snapshot_purpose':'Immutable initialization controls; not resumable optimizer checkpoints'}
    atomic_json(path,info);atomic_json(Path(artifact)/'initialization_manifest.json',info)
    return models,info

_strict_training_protocol=_BASE._strict_training_protocol

def _plan_binding(root,plan,bundle):
    safe=lambda value:isinstance(value,str) and bool(value) and all(c.isalnum() or c in '-_' for c in value)
    if not safe(plan.get('run_id')) or not safe(plan.get('stage_id')):raise ValueError('Safe run and stage identifiers required')
    arms=plan.get('arms',[])
    if not arms or len({a['id'] for a in arms})!=len(arms) or any(not safe(a['id']) for a in arms):raise ValueError('Unique safe arm identifiers required')
    updates=int(plan['updates']);milestones=list(map(int,plan.get('milestones',[updates])))
    if updates<4 or milestones!=sorted(set(milestones)) or not milestones or any(m<1 or m>updates for m in milestones) or milestones[-1]!=updates:raise ValueError('Explicit ordered milestones ending at full schedule required')
    if int(plan.get('round_updates',100))<1:raise ValueError('Positive round size required')
    train=list(bundle['train_ids']);probe=list(bundle['probe_train_ids']);dev=list(bundle['dev_ids']);heldout=list(bundle.get('heldout_ids',[]))
    if not train or not probe or not dev or any(len(ids)!=len(set(ids)) for ids in (train,probe,dev)):raise ValueError('Unique nonempty memberships required')
    if set(train)&set(dev) or set(probe)&set(dev) or (set(train)|set(probe)|set(dev))&set(heldout):raise ValueError('Development/official-test leakage')
    if not set(probe).issubset(train):raise ValueError('Probe training membership must be nested in stage training')
    for name,ids in [('train_count',train),('probe_train_count',probe)]:
        if name in plan and int(plan[name])!=len(ids):raise ValueError('Registered membership count differs:'+name)
    seeds=sorted({int(a['seed']) for a in arms})
    if 'seeds' in plan and sorted(map(int,plan['seeds']))!=seeds:raise ValueError('Declared seed membership differs')
    if 'registered_seeds' in plan and not set(seeds).issubset(set(map(int,plan['registered_seeds']))):raise ValueError('Unregistered model seeds')
    if tuple(plan.get('variants',VARIANTS))!=VARIANTS:raise ValueError('Exactly ViT and one optional geometry candidate required')
    if plan.get('probe_endpoints') not in (['initial','final'],['final']):raise ValueError('Registered probe endpoints required')
    if not 0<float(plan.get('hard_limit_seconds',10800))<=43200:raise ValueError('Positive bounded runtime required')
    for arm in arms:
        config=arm['config'];policy=_strict_training_protocol(config);variant=arm.get('variant',arm.get('condition'))
        if variant not in VARIANTS or arm['kind']!=('vit' if variant=='vit' else 'semantic') or int(config['seeds']['model_init'])!=int(arm['seed']):raise ValueError('Architecture/variant/seed mismatch')
        if policy['updates']!=updates or policy['effective_batch']!=int(plan.get('effective_batch',128)):raise ValueError('Schedule/effective batch differs')
        if config.get('dtype','float32')!='float32' or config.get('evaluation',{}).get('heldout_enabled',False):raise ValueError('FP32 and closed official test required')
        if policy['checkpoint_every']>50:raise ValueError('Checkpoint interval must be at most50 successful updates')
        if int(config.get('cpu_threads',2))!=2:raise ValueError('Registered deterministic configuration requires2 CPU threads')
    for seed in seeds:
        group=[a for a in arms if int(a['seed'])==seed]
        if sorted(a.get('variant',a.get('condition')) for a in group)!=sorted(VARIANTS):raise ValueError('Each seed needs precisely the two registered arms')
        configs=[]
        for arm in group:
            config=normalize_stage_plan(arm['config']);options=config.pop('kernel15',None)
            if not options or options.get('variant')!=arm.get('variant',arm.get('condition')):raise ValueError('Explicit kernel15 variant differs')
            options={k:v for k,v in options.items() if k!='variant'};config['kernel15_common_options']=options;config.get('model',{}).pop('kind',None);configs.append(config)
        if configs[0]!=configs[1]:raise ValueError('Paired configurations differ beyond registered architecture choice')
    actual=stage_definition_hashes(root);sources=plan.get('source_hashes',actual)
    if sources!=actual:raise ValueError('Definition source hashes changed')
    consumer=plan.get('consumer_notebook_sha256')
    if consumer:
        notebook=json.loads((Path(root)/plan.get('resume_notebook',RESUME_NOTEBOOK)).read_text());code=[''.join(c['source']) for c in notebook['cells'] if c['cell_type']=='code']
        if _hash(code)!=consumer:raise ValueError('Registered front-door notebook code changed')
    return {'schema':'fmm-jepa-notebook15-stage-binding/v1','plan_hash':_hash(plan),'source_hashes':sources,
        'dataset_fingerprint':bundle['dataset'].fingerprint,'split_sha256':bundle['split_manifest']['split_sha256'],
        'training_ids_sha256':_hash(train),'training_examples':len(train),'probe_training_ids_sha256':_hash(probe),'probe_training_examples':len(probe),
        'development_ids_sha256':_hash(dev),'development_examples':len(dev),'heldout_status':'NOT_RUN'}

def _persist_numerical_backend_stats(artifact,stage_id,arm_id,event):
    value={'status':'RECORDED','backend':'ViT SDPA plus optional signed spatial pair bias','stage_id':stage_id,'arm_id':arm_id,'event':event,
           'updated_unix':time.time(),'worker_pid':os.getpid(),'bound_builder_sources':dict(_BOUND_SOURCE_IDENTITY)}
    atomic_json(Path(artifact)/'numerical_backend_stats.json',value);return value

for _name in ('build_model','initialize_models','stage_definition_hashes','_plan_binding','_persist_numerical_backend_stats'):
    setattr(_BASE,_name,globals()[_name])
_BASE.RESUME_NOTEBOOK=RESUME_NOTEBOOK;_BASE.RESUME_CELL=RESUME_CELL
for _name in ('_rng','_restore_rng','_atomic_torch','_record','_guard','_deadline','_status','_registered','_arm_paths','_initial_model','_release_semantic','validate_training_manifest'):
    globals()[_name]=getattr(_BASE,_name)
_BASE_RUN_STAGE_MODEL=_BASE._run_stage_model

def _run_stage_model(plan,*args,**kwargs):
    config=args[1] if len(args)>1 else kwargs['config'];configure_environment(config)
    return _BASE_RUN_STAGE_MODEL(plan,*args,**kwargs)
_BASE._run_stage_model=_run_stage_model


def verify_stage_resume(root,artifact,plan,bundle,deadline_unix=None):
    """At user execution only: actual-config four updates equals reconstructed2+2.

    One first-seed certificate per architecture covers that stage resolution,
    batch/accumulation, data membership and full successful-update schedule.
    Every RNG/optimizer/model/scheduler/EMA/sampler/trace is compared bitwise;
    elapsed timing is the only excluded state.
    """
    from fmm_jepa.food101_benchmark import _food_exact_tree
    root,artifact=Path(root),Path(artifact);plan=normalize_stage_plan(plan)
    registration=_registered(root,artifact,plan,bundle);path=artifact/'stage_preflight.json'
    if path.exists():
        certificate=json.loads(path.read_text())
        if certificate['status']!='PASS' or certificate['binding']!=registration['binding']:
            raise ValueError('Actual-config preflight identity differs')
        expected={next(a['id'] for a in plan['arms'] if a['variant']==variant) for variant in plan['variants']}
        if set(certificate['arms'])!=expected:raise ValueError('Resume certificate must cover every registered variant')
        for entry in certificate['arms'].values():
            for record in entry['checkpoints'].values():
                folder=Path(record['directory']);manifest=validate_training_manifest(folder)
                if manifest['successful_updates']!=4 or digest_file(folder/manifest['latest'])!=record['sha256']:
                    raise ValueError('Full-state preflight evidence changed')
        return certificate
    selected=[]
    for variant in plan['variants']:
        selected.append(next(a for a in plan['arms'] if a['variant']==variant))
    fields=['model','optimizer','scheduler','ema_position','scaler','precision','rng','sampler',
            'streams','microstep','partial_accumulation','world_size','rank','replay','config','protocol','identity']
    state_fields=['successful_updates','attempted_updates','image_exposures','consecutive_failures','traces','failures','diagnostics']
    timing={'data_seconds','compute_seconds','step_seconds','images_per_second','_cuda_event_count'}
    certificate={'status':'RUNNING','binding':registration['binding'],'arms':{},
        'scope':'First seed per registered variant; exact actual stage config/data and full schedule; elapsed timing excluded'}
    original_rng=_rng()
    try:
        with _guard(deadline_unix):
            for arm in selected:
                _deadline(deadline_unix,5);_status(artifact,'exact_resume','RUNNING',arm_id=arm['id'])
                arm_folder,_=_arm_paths(root,artifact,plan,arm)
                protocol=json.loads((arm_folder/'protocol.json').read_text())
                folder=artifact/'preflight'/arm['id'];folder.mkdir(parents=True,exist_ok=True)
                rng_path=folder/'trajectory_rng.pt';rng_record=folder/'trajectory_rng.json'
                if rng_record.exists():
                    if digest_file(rng_path)!=json.loads(rng_record.read_text())['sha256']:raise ValueError('Preflight RNG changed')
                    trajectory=torch.load(rng_path,map_location='cpu',weights_only=False)
                else:
                    if rng_path.exists():raise ValueError('Orphan trajectory RNG; inspect before recovery')
                    with isolated_rng(int(arm['seed'])):trajectory=_rng()
                    _atomic_torch(rng_path,trajectory);atomic_json(rng_record,_record(rng_path))
                reference_dir=root/'checkpoints'/plan['run_id']/'stages'/plan['stage_id']/'preflight'/(arm['id']+'-reference')
                split_dir=reference_dir.with_name(arm['id']+'-split')
                _restore_rng(copy.deepcopy(trajectory))
                reference=_run_stage_model(plan,arm['kind'],arm['config'],bundle,folder/'reference',reference_dir,protocol,
                    model=_initial_model(root,artifact,plan,arm),stop_after=4,deadline_unix=deadline_unix)
                del reference
                _restore_rng(copy.deepcopy(trajectory))
                partial=_run_stage_model(plan,arm['kind'],arm['config'],bundle,folder/'split',split_dir,protocol,
                    model=_initial_model(root,artifact,plan,arm),stop_after=2,deadline_unix=deadline_unix)
                if partial['resume']['final_step'] not in (2,4):raise AssertionError('Preflight cutoff before split boundary')
                del partial
                resumed=_run_stage_model(plan,arm['kind'],arm['config'],bundle,folder/'split',split_dir,protocol,
                    model=_initial_model(root,artifact,plan,arm),stop_after=4,deadline_unix=deadline_unix)
                lm,rm=validate_training_manifest(reference_dir),validate_training_manifest(split_dir)
                left=torch.load(reference_dir/lm['latest'],map_location='cpu',weights_only=False)
                right=torch.load(split_dir/rm['latest'],map_location='cpu',weights_only=False)
                for field in fields:_food_exact_tree(left[field],right[field],field)
                for field in state_fields:_food_exact_tree(left['state'][field],right['state'][field],'state.'+field)
                lmetrics=[{k:v for k,v in row.items() if k not in timing} for row in left['state']['metrics']]
                rmetrics=[{k:v for k,v in row.items() if k not in timing} for row in right['state']['metrics']]
                _food_exact_tree(lmetrics,rmetrics,'deterministic_metrics')
                if lm['successful_updates']!=4 or rm['attempted_updates']!=4 or len(rm['checkpoints'])<2:
                    raise AssertionError('Four finite updates and latest-two retention required')
                certificate['arms'][arm['id']]={'status':'PASS','bitwise_equal':True,
                    'image_size':arm['config']['image_size'],'training_examples':len(bundle['train_ids']),
                    'microbatch':arm['config']['training']['microbatch'],'accumulation_steps':arm['config']['training']['accumulation_steps'],
                    'schedule_updates':plan['updates'],'full_state_fields':fields,
                    'full_jepa_step_seconds':resumed['performance'].get('complete_jepa_step_seconds'),
                    'jepa_compute_seconds':resumed['performance'].get('jepa_compute_seconds'),
                    'peak_cuda_allocated_bytes':resumed['performance'].get('peak_cuda_allocated_bytes'),
                    'performance_scope':'Tiny exact-resume preflight only; timing sample count small, not a scientific efficiency comparison',
                    'checkpoints':{'reference':{'directory':str(reference_dir),'sha256':digest_file(reference_dir/lm['latest'])},
                                   'resumed':{'directory':str(split_dir),'sha256':digest_file(split_dir/rm['latest'])}}}
                atomic_json(folder/'certificate.json',certificate['arms'][arm['id']])
                del left,right,resumed;_release_semantic(arm['config']['device'])
    finally:_restore_rng(original_rng)
    certificate['status']='PASS';atomic_json(path,certificate)
    return certificate
_BASE.verify_stage_resume=verify_stage_resume


def prepare_stage(root,artifact,plan,bundle):
    from fmm_jepa.food101_evaluation import encoder_state_hash
    root,artifact=Path(root),Path(artifact);plan=normalize_stage_plan(plan)
    artifact.mkdir(parents=True,exist_ok=True);binding=_plan_binding(root,plan,bundle)
    path=artifact/'stage_registration.json'
    if path.exists():
        registration=json.loads(path.read_text())
        if registration['binding']!=binding or registration['normalized_plan']!=plan:raise ValueError('Existing registration differs; no silent restart')
    else:registration={'binding':binding,'normalized_plan':plan,'arms':{},'heldout_status':'NOT_RUN'}
    paired_shared={};paired_kernel_mixer={}
    for arm in plan['arms']:
        folder,checkpoint=_arm_paths(root,artifact,plan,arm);folder.mkdir(parents=True,exist_ok=True)
        models,initial=initialize_models(arm['config'],folder);model=models[arm['kind']]
        shared=initial['shared_parameter_hashes'][arm['kind']];seed=int(arm['seed'])
        if seed in paired_shared and paired_shared[seed]!=shared:raise AssertionError('Paired common initial parameters differ')
        paired_shared[seed]=shared
        mixer_hash=initial.get('kernel_mixer_shared_sha256')
        if mixer_hash is not None:
            if seed in paired_kernel_mixer and paired_kernel_mixer[seed]!=mixer_hash:raise AssertionError('Paired kernel value/output/bandwidth/mixture initialization differs')
            paired_kernel_mixer[seed]=mixer_hash
        parameter_hash=encoder_state_hash({name:value.detach().cpu() for name,value in model.named_parameters() if value.requires_grad})
        protocol={'schema':'fmm-jepa-notebook15-arm/v1','binding':binding,'source_hashes':binding['source_hashes'],
            'arm':arm,'initialization':initial,'training':_strict_training_protocol(arm['config']),
            'milestones':plan['milestones'],'selection':'Predeclared final full-state checkpoint; fixed probe epoch20',
            'heldout_status':'NOT_RUN'}
        protocol_path=folder/'protocol.json'
        if protocol_path.exists() and json.loads(protocol_path.read_text())!=protocol:raise ValueError('Arm protocol changed')
        atomic_json(protocol_path,protocol)
        registration['arms'][arm['id']]={'protocol_sha256':digest_file(protocol_path),'initialization':initial,
            'initial_teacher':initial['states'][arm['kind']]['teacher'],'trainable_parameter_hash':parameter_hash,
            'shared_parameter_hash':shared,'kernel_mixer_shared_sha256':mixer_hash,'variant':arm['variant'],
            'checkpoint_directory':str(checkpoint.resolve())}
        del model,models;gc.collect()
    atomic_json(path,registration)
    return registration
_BASE.prepare_stage=prepare_stage


def calibrate_case(root,artifact,plan,arm,bundle,stop_after=8,deadline_unix=None):
    """Discarded actual full-update calibration; never supplies a quality score."""
    artifact=Path(artifact);artifact.mkdir(parents=True,exist_ok=True)
    models,initial=initialize_models(arm['config'],artifact)
    model=models[arm['kind']]
    del models
    protocol={'schema':'kernel15-discarded-calibration/v1','arm':arm,
        'source_hashes':stage_definition_hashes(root),'initialization':initial,
        'stop_after':int(stop_after),'quality_selection':'FORBIDDEN','heldout_status':'NOT_RUN'}
    checkpoint=Path(root)/'checkpoints'/plan['run_id']/'calibration'/arm['id']
    result=_run_stage_model(plan,arm['kind'],arm['config'],bundle,artifact,checkpoint,protocol,
        model=model,stop_after=int(stop_after),deadline_unix=deadline_unix)
    if result['resume']['final_step']!=int(stop_after) or any(not r.get('successful',False) for r in result['traces']):
        raise RuntimeError('Calibration must complete all finite updates')
    _release_semantic(arm['config']['device'])
    return result

def _software_fixture(plan):
    return (plan.get('evidence_role')=='SYNTHETIC_SOFTWARE_VALIDATION_ONLY' and int(plan['updates'])<=4 and
        float(plan.get('hard_limit_seconds',float('inf')))<=300 and all(a['config'].get('run_mode')=='software_validation' and
        a['config'].get('device')=='cpu' for a in plan['arms']))

# Adapt only the inherited tmux guard for explicitly tiny CPU software fixtures.
# Read only tagged definition source; no upstream invocation or training cells run.
import ast as _ast
_source='\n'.join(''.join(c['source']) for c in json.loads((ROOT/'experiments/05_scaling/runner.ipynb').read_text())['cells']
    if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[]))
_original_train_source=next(_ast.get_source_segment(_source,node) for node in _ast.parse(_source).body
    if isinstance(node,_ast.FunctionDef) and node.name=='train_stage')
_original_guard="if not os.environ.get('TMUX'):raise RuntimeError('Long stage training requires detached tmux')"
if _original_guard not in _original_train_source:raise ValueError('Inherited training guard changed; inspect before adapting')
_fixture_train_source=_original_train_source.replace(_original_guard,"if not os.environ.get('TMUX') and not _software_fixture(plan):raise RuntimeError('Long stage training requires detached tmux')")
_BASE._software_fixture=_software_fixture
exec(compile(_fixture_train_source,'runner05#explicit-software-fixture-guard','exec'),_BASE.__dict__)

def train_stage(root,artifact,plan,bundle,deadline_unix=None):
    configure_environment(plan['arms'][0]['config'])
    return _BASE.train_stage(root,artifact,plan,bundle,deadline_unix)
validate_stage_milestones=_BASE.validate_stage_milestones
train_calibration=calibrate_case
